# skin-lesion-ai — GPU training (HAM10000)

Runs on Kaggle with **GPU + Internet** enabled. Clones the repo, installs it, prepares the attached dataset, trains, evaluates, and writes results into `artifacts/` (saved as the kernel Output).

In [ ]:
!nvidia-smi -L || echo 'no GPU'

In [ ]:
REPO = 'skin-lesion-ai'
import os, subprocess
if not os.path.isdir(f'/kaggle/working/{REPO}'):
    subprocess.run(['git','clone','--depth','1','https://github.com/sara-tavakoli/'+REPO+'.git'], cwd='/kaggle/working', check=True)
os.chdir(f'/kaggle/working/{REPO}')
subprocess.run(['pip','-q','install','-e','.'], check=True)
import importlib, skinlesion; print('skinlesion OK')

## 1 · Prepare dataset

In [ ]:
# --- normalise the Kaggle HAM10000 layout into data/ham10000/{metadata.csv, images/} ---
import shutil, os, pathlib, pandas as pd
SRC = pathlib.Path("/kaggle/input/skin-cancer-mnist-ham10000")
assert SRC.is_dir(), f"attach the dataset kmader/skin-cancer-mnist-ham10000 (missing {SRC})"
meta_csv = next((p for p in SRC.rglob("*metadata*.csv")), None) \
        or next((p for p in SRC.rglob("HAM10000_metadata*")), None)
assert meta_csv, "HAM10000 metadata csv not found in the attached dataset"
OUT = pathlib.Path("data/ham10000"); (OUT / "images").mkdir(parents=True, exist_ok=True)
meta = pd.read_csv(meta_csv)
keep = [c for c in ["image_id","lesion_id","dx","dx_type","age","sex","localization"] if c in meta.columns]
meta[keep].to_csv(OUT / "metadata.csv", index=False)
jpegs = {p.stem: p for p in SRC.rglob("*.jpg")}
missing = 0
for iid in meta["image_id"]:
    src = jpegs.get(iid)
    if src is None:
        missing += 1; continue
    dst = OUT / "images" / f"{iid}.jpg"
    if not dst.exists():
        try: os.symlink(src, dst)
        except OSError: shutil.copy2(src, dst)
print("images:", len(list((OUT/'images').glob('*.jpg'))), "| missing:", missing)
print(meta["dx"].value_counts().to_string())

## 2 · Train

In [ ]:
!python scripts/prepare_splits.py --data-dir data/ham10000
!python scripts/train.py --experiment focal_balanced \
    data.image_size=256 data.batch_size=32 data.num_workers=2 \
    train.max_epochs=40 train.precision=16-mixed

## 3 · Evaluate

In [ ]:
!python scripts/evaluate.py --checkpoint artifacts/best.ckpt --n-bootstrap 2000
!python scripts/explain.py --checkpoint artifacts/best.ckpt --images data/ham10000/images --limit 24 --out artifacts/cams

## 4 · Show results

In [ ]:
import pathlib, IPython.display as D
for md in sorted(pathlib.Path('.').rglob('RESULTS.md')):
    D.display(D.Markdown(md.read_text()))
for png in sorted(pathlib.Path('artifacts').rglob('*.png'))[:16]:
    print(png); D.display(D.Image(str(png)))